# Bangladesh Food Price Watch — prediction system resume
Source: the repository notebook and the 2026-11-15 handover. The private Drive notebook was not accessible; its saved outputs are not copied here.

Push this notebook **and the merged source code** to GitHub before opening it in Colab.
Use the Google account holding `MyDrive/FoodPriceWatch` (handover: mehedi11908015@gmail.com).
Choose a GPU runtime if desired; the current scripts use their existing CPU settings.

Run setup, then **one stage cell at a time**. Paste its full output back to the agent and stop for review.
Do not use Run all. This notebook does not run the earlier benchmark or sealed tests.

Remaining: verify S00/S22/S23 on Drive; implement S24 stacked model; confirm S25 candidates and validation period; selection S25; tuning S26; committed pre-registration and one sealed test S27; ranges/alerts S28; explanations S29; production fit/exports S30; dashboard S31; update script S32; docs S33. Report writing follows verified results.


## 1. Mount Drive
Authorize the account containing the existing project data. No temporary-storage fallback is used.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, json, hashlib, csv
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/FoodPriceWatch')
assert Path('/content/drive/MyDrive').is_dir(), 'STOP: Drive is not mounted'
print('Project root:', ROOT)
print('Existing project folder:', ROOT.exists())


## 2. Load code
An existing checkout must be clean. Updates use fast-forward only and stop on divergence, preserving Drive run commits. Send any failure to the agent; do not reset the checkout.


In [ ]:
REPO = 'https://github.com/mrchowdhury587/foodprice_watch.git'
def git(*args):
    return subprocess.check_output(['git', *args], cwd=ROOT, text=True).strip()
if not (ROOT / '.git').exists():
    assert not ROOT.exists() or not any(ROOT.iterdir()), 'STOP: existing non-Git folder; ask the agent to attach code without replacing data'
    ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', REPO, str(ROOT)], check=True)
else:
    assert not git('status', '--porcelain'), 'STOP: uncommitted Drive changes; paste git status to the agent'
    assert git('branch', '--show-current') == 'main', 'STOP: unexpected branch'
    subprocess.run(['git', 'fetch', 'origin', 'main'], cwd=ROOT, check=True)
    subprocess.run(['git', 'merge', '--ff-only', 'origin/main'], cwd=ROOT, check=True)
for name in ['src/fpw/features.py', 'src/stages/s23_features.py']:
    assert (ROOT / name).is_file(), 'STOP: push merged feature code to GitHub first: ' + name
subprocess.run(['git', 'config', 'user.name', 'FPW runner'], cwd=ROOT, check=True)
subprocess.run(['git', 'config', 'user.email', 'fpw@example.com'], cwd=ROOT, check=True)
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src')
sys.path.insert(0, str(ROOT / 'src'))
print('Code commit:', git('rev-parse', 'HEAD'))


## 3. Install runtime packages


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements-colab.txt')], check=True)
os.environ.setdefault('FPW_THREADS', str(os.cpu_count() or 2))
print('Package install: PASS; threads:', os.environ['FPW_THREADS'])


## 4. Startup verification
A failed check stops execution. Missing or changed finished outputs are reset to pending by the project checker; paste failures back before continuing.


In [ ]:
for folder in ['outputs', 'data']:
    for tmp in (ROOT / folder).rglob('*.tmp'):
        assert tmp.resolve().is_relative_to(ROOT.resolve())
        tmp.unlink()
        print('Removed interrupted-write temporary file:', tmp.relative_to(ROOT))
subprocess.run([sys.executable, '-m', 'fpw.check'], cwd=ROOT, check=True)
def run_unit(sid):
    subprocess.run([sys.executable, '-m', 'fpw.runner', sid], cwd=ROOT, check=True)
    ledger = json.loads((ROOT / 'state/stages.json').read_text())
    stage = next(s for s in ledger['stages'] if s['id'] == sid)
    assert stage['status'] == 'done', 'STOP: stage was not finalized'
    summary = []
    for name, expected in stage['output_sha256'].items():
        path = ROOT / name
        actual = hashlib.sha256(path.read_bytes()).hexdigest()
        assert actual == expected, 'STOP: output hash mismatch: ' + name
        rows = None
        if path.suffix == '.csv':
            with path.open(newline='') as f:
                rows = sum(1 for _ in csv.DictReader(f))
        elif path.suffix == '.parquet':
            import pyarrow.parquet as pq
            rows = pq.read_metadata(path).num_rows
        summary.append(dict(file=name, rows=rows, sha256=actual))
    assert summary, 'STOP: no finalized output hashes'
    print(json.dumps(dict(unit=sid, outputs=summary, check='PASS', git_commit=git('rev-parse', 'HEAD')), indent=2))


## 5. S00 — refresh intake
Before running, place the extended CSV `tcb_dhaka_daily_prices_2021_2026_v2.csv` in `inbox/project/`.
Existing original project files must be in `data/raw/project/` or `inbox/project/`.
This refreshes the intake manifest and its finalized hashes. Paste the output back and stop.


In [ ]:
subprocess.run([sys.executable, 'src/stages/s00_intake.py'], cwd=ROOT, check=True)
subprocess.run([sys.executable, 'src/stages/finalize.py', 'S00'], cwd=ROOT, check=True)
run_unit('S00')


## 6. S22 — extended panel
Run after S00 output is reviewed. The earlier S01 panel must already exist on Drive for the equality checks. Audit tables from the work-PC audit must be available. Missing prerequisites require agent review; do not rerun the old benchmark.


In [ ]:
run_unit('S22')


## 7. S23 — kept features
Run after S22 output is reviewed. Requires the earlier driver artifacts and feature table for comparison. Paste the output back and stop.


In [ ]:
run_unit('S23')


## 8. Inspect remaining work — no training or testing here
S24–S33 are not implemented by this notebook. Their pending status is not evidence of completed training.
Before S25, confirm candidates/winner rule and validation period (`TO_CONFIRM` in config/project.yaml).
The test must follow committed pre-registration and the existing seal mechanism. Never tune on test results.
Keep data, models and exports out of GitHub.


In [ ]:
ledger = json.loads((ROOT / 'state/stages.json').read_text())
for stage in ledger['stages']:
    if stage['id'] in {f'S{i}' for i in range(22, 34)}:
        print(stage['id'], stage['status'], '-', stage['title'])
print('Unconfirmed settings:')
for line in (ROOT / 'config/project.yaml').read_text().splitlines():
    if 'TO_CONFIRM' in line and not line.lstrip().startswith('#'):
        print(line)
